# d=10 Configuration Parity Check (mPCN vs MESS setup)

This notebook compares key solute-transport configuration values between:
- mcmc-internal notebook 04 (mPCN rho/P sweep), and
- mess-internal notebook 11 (MESS shared-draw sweep).

It focuses on the d=10 setup and reports any mismatch before combining reused mPCN results with new MESS outputs.

In [2]:
import ast
import json
import re
from pathlib import Path

import pandas as pd

In [3]:
def resolve_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent and not (cur / 'pyproject.toml').exists():
        cur = cur.parent
    return cur

mess_internal_root = resolve_repo_root(Path.cwd())
workspace_root = mess_internal_root.parent

mcmc_nb_path = workspace_root / 'mcmc-internal' / 'notebooks' / '04_toy_solute_transport_mpcn_P_rho_sweep.ipynb'
mess_nb_path = mess_internal_root / 'notebooks' / '11_advection_diffusion_dim_sweep_shared_draws.ipynb'

print('mcmc notebook:', mcmc_nb_path)
print('mess notebook:', mess_nb_path)
print('mcmc exists:', mcmc_nb_path.exists())
print('mess exists:', mess_nb_path.exists())

mcmc notebook: /Users/guillers/Documents/GitHub/mcmc/mcmc-internal/notebooks/04_toy_solute_transport_mpcn_P_rho_sweep.ipynb
mess notebook: /Users/guillers/Documents/GitHub/mcmc/mess-internal/notebooks/11_advection_diffusion_dim_sweep_shared_draws.ipynb
mcmc exists: True
mess exists: True


In [4]:
def load_notebook_code_text(path: Path) -> str:
    payload = json.loads(path.read_text(encoding='utf-8'))
    lines = []
    for cell in payload.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        for raw in cell.get('source', []):
            lines.append(raw.rstrip('\n'))
    return '\n'.join(lines)

def _strip_inline_comment(line: str) -> str:
    if '#' not in line:
        return line
    parts = line.split('#', 1)
    return parts[0].rstrip()

def extract_literal_assignments(code_text: str, keys: list[str]) -> dict:
    out = {}
    for key in keys:
        pattern = re.compile(rf'^\s*{re.escape(key)}\s*=\s*(.+)$', re.MULTILINE)
        match = pattern.search(code_text)
        if not match:
            out[key] = None
            continue
        rhs = _strip_inline_comment(match.group(1).strip())
        try:
            out[key] = ast.literal_eval(rhs)
        except Exception:
            out[key] = rhs
    return out

mcmc_text = load_notebook_code_text(mcmc_nb_path)
mess_text = load_notebook_code_text(mess_nb_path)

In [5]:
mcmc_keys = [
    'seed_data', 'seed_mcmc', 'd', 'kappa', 'sigma', 'alpha', 'gamma', 'tau2',
    'a_mode', 'use_prior_A', 'shared_draws_seed', 'obs_highest_freq', 'obs_bandwidth', 'obs_config'
]
mess_keys = [
    'seed_data', 'seed_mcmc', 'd_list', 'd_max', 'kappa', 'sigma', 'alpha', 'gamma', 'tau2',
    'a_mode', 'use_prior_A', 'shared_draws_seed', 'obs_highest_freq', 'obs_bandwidth', 'obs_config'
]

mcmc_cfg = extract_literal_assignments(mcmc_text, mcmc_keys)
mess_cfg = extract_literal_assignments(mess_text, mess_keys)

mess_d10 = None
if isinstance(mess_cfg.get('d_list'), (list, tuple)) and mess_cfg['d_list']:
    mess_d10 = mess_cfg['d_list'][0]

comparison_rows = []
for key in [
    'seed_data', 'seed_mcmc', 'kappa', 'sigma', 'alpha', 'gamma', 'tau2',
    'a_mode', 'use_prior_A', 'shared_draws_seed', 'obs_highest_freq', 'obs_bandwidth', 'obs_config'
]:
    left = mcmc_cfg.get(key)
    right = mess_cfg.get(key)
    comparison_rows.append({
        'key': key,
        'mcmc_internal_notebook04': left,
        'mess_internal_notebook11': right,
        'match': left == right,
    })

comparison_rows.append({
    'key': 'd (mcmc) vs first d in d_list (mess)',
    'mcmc_internal_notebook04': mcmc_cfg.get('d'),
    'mess_internal_notebook11': mess_d10,
    'match': mcmc_cfg.get('d') == mess_d10,
})

comparison_rows.append({
    'key': 'd_max (informational)',
    'mcmc_internal_notebook04': 'not explicit in top config cell',
    'mess_internal_notebook11': mess_cfg.get('d_max'),
    'match': None,
})

df = pd.DataFrame(comparison_rows)
display(df)

,key,mcmc_internal_notebook04,mess_internal_notebook11,match
0,seed_data,0,0,True
1,seed_mcmc,0,0,True
2,kappa,0.02,0.02,True
3,sigma,0.5,0.5,True
4,alpha,3.0,3,True
5,gamma,2.0,2,True
6,tau2,2.0,2.0,True
7,a_mode,nearest_neighbor,nearest_neighbor,True
8,use_prior_A,True,True,True
9,shared_draws_seed,seed_data,seed_data,True


In [6]:
hard_mismatches = df[df['match'] == False]
if hard_mismatches.empty:
    print('Parity check status: PASS for compared top-level keys.')
else:
    print('Parity check status: MISMATCH detected on top-level keys.')
    display(hard_mismatches[['key', 'mcmc_internal_notebook04', 'mess_internal_notebook11']])

print('Reminder: this check compares explicit literal assignments only.')
print('If deeper generator logic differs, keep a run-level parity report (config.json + data hash + run id).')

Parity check status: PASS for compared top-level keys.
Reminder: this check compares explicit literal assignments only.
If deeper generator logic differs, keep a run-level parity report (config.json + data hash + run id).
